# Feature Engineering — CA_1 Retail Demand Forecasting

## Objective

This notebook transforms the processed CA_1 retail dataset into a
model-ready forecasting dataset.

Feature engineering is guided by the patterns identified during exploratory
data analysis, including weekly and monthly seasonality, intermittent product
demand, calendar events, SNAP effects, product availability, and relative
price behavior.

The feature set will include:

- calendar and seasonal features;
- product hierarchy features;
- lagged demand features;
- rolling demand statistics;
- price and promotion-related features;
- event and SNAP indicators;
- product availability information.

## Leakage Prevention

All forecasting features must represent information that would be available
at the time a prediction is generated.

In particular, lagged and rolling demand features will be constructed using
past observations only. Future sales information must never be included in
the predictors.

Model validation will also preserve chronological ordering rather than using
a random train-test split.

In [1]:
# ---------------------------------------------------------
# Import core libraries
# ---------------------------------------------------------
# Pathlib is used to build operating-system-independent paths.
# NumPy and pandas will support feature construction and data
# manipulation throughout the notebook.

from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
# ---------------------------------------------------------
# Define project paths
# ---------------------------------------------------------
# The notebook is stored inside the notebooks/ directory, so
# the project root is one level above the current directory.

PROJECT_ROOT = Path.cwd().parent

PROCESSED_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

processed_file = (
    PROCESSED_DATA_DIR
    / "ca1_model_data.parquet"
)

print("Project root:", PROJECT_ROOT)
print("Processed file:", processed_file)

Project root: C:\Users\saman\OneDrive\Desktop\Projects\retail-demand-intelligence
Processed file: C:\Users\saman\OneDrive\Desktop\Projects\retail-demand-intelligence\data\processed\ca1_model_data.parquet


In [3]:
# ---------------------------------------------------------
# Load the processed CA_1 dataset
# ---------------------------------------------------------
# This Parquet file contains the cleaned item-day forecasting
# grain created during the data preparation phase:
#
# one product × one store × one day.
#
# Loading from the checkpoint avoids repeating the expensive
# wide-to-long transformation and joins performed earlier.

df = pd.read_parquet(
    processed_file,
    engine="pyarrow"
)

print(
    "Dataset shape:",
    f"{df.shape[0]:,} rows × {df.shape[1]} columns"
)

print(
    "Date range:",
    df["date"].min(),
    "to",
    df["date"].max()
)

print(
    "Products:",
    f"{df['item_id'].nunique():,}"
)

Dataset shape: 5,918,109 rows × 21 columns
Date range: 2011-01-29 00:00:00 to 2016-05-22 00:00:00
Products: 3,049


In [4]:
# ---------------------------------------------------------
# Validate the modeling grain
# ---------------------------------------------------------
# There should be exactly one row for each
# item × store × date combination.
#
# Duplicate observations would cause incorrect lag and rolling
# calculations later, so we verify the grain before generating
# any time-series features.

duplicate_rows = (
    df.duplicated(
        subset=[
            "item_id",
            "store_id",
            "date",
        ]
    )
    .sum()
)

print(
    "Duplicate item-store-date rows:",
    f"{duplicate_rows:,}"
)

print(
    "Missing sales values:",
    f"{df['sales'].isna().sum():,}"
)

print(
    "Missing dates:",
    f"{df['date'].isna().sum():,}"
)

Duplicate item-store-date rows: 0
Missing sales values: 0
Missing dates: 0


In [5]:
# ---------------------------------------------------------
# Inspect existing columns before feature construction
# ---------------------------------------------------------
# The processed dataset already contains several calendar
# fields inherited from the M5 calendar table.
#
# Reviewing them first prevents us from creating unnecessary
# duplicate variables.

print("Current columns:")
for column in df.columns:
    print(f" - {column}")

Current columns:
 - id
 - item_id
 - dept_id
 - cat_id
 - store_id
 - state_id
 - d
 - sales
 - date
 - wm_yr_wk
 - weekday
 - wday
 - month
 - year
 - event_name_1
 - event_type_1
 - event_name_2
 - event_type_2
 - snap_CA
 - sell_price
 - is_available


## 1. Calendar and Seasonal Features

The processed dataset already contains several calendar variables from the M5
calendar, including weekday, month, year, event information, and the California
SNAP indicator.

Additional calendar features are created to represent recurring temporal
patterns identified during exploratory analysis.

The original detailed calendar variables are retained so that the forecasting
models can distinguish between general calendar effects and specific events.

In [6]:
# ---------------------------------------------------------
# Create additional calendar features
# ---------------------------------------------------------
# Existing month, year, weekday, and M5 week identifiers are
# retained rather than recreated from the date column.

df["day_of_month"] = (
    df["date"]
    .dt.day
    .astype("uint8")
)

df["week_of_year"] = (
    df["date"]
    .dt.isocalendar()
    .week
    .astype("uint8")
)

df["quarter"] = (
    df["date"]
    .dt.quarter
    .astype("uint8")
)


# ---------------------------------------------------------
# Create weekend indicator
# ---------------------------------------------------------
# EDA showed substantially higher CA_1 demand on Saturday
# and Sunday than during the middle of the week.
#
# Using the date-derived weekday number avoids depending on
# text labels when constructing this binary feature.
# Monday = 0 and Sunday = 6.

df["is_weekend"] = (
    df["date"].dt.dayofweek >= 5
).astype("uint8")


# ---------------------------------------------------------
# Create general event indicator
# ---------------------------------------------------------
# Detailed event_name and event_type columns are retained
# because EDA showed that individual events behave very
# differently.
#
# This additional binary variable simply indicates whether
# at least one calendar event occurs on the date.

df["is_event"] = (
    df["event_name_1"].notna()
    | df["event_name_2"].notna()
).astype("uint8")

In [7]:
# ---------------------------------------------------------
# Validate calendar feature ranges
# ---------------------------------------------------------
# These checks help catch incorrect date extraction or type
# conversion before more complex time-series features are
# constructed.

calendar_feature_summary = pd.DataFrame(
    {
        "min": df[
            [
                "day_of_month",
                "week_of_year",
                "quarter",
                "is_weekend",
                "is_event",
            ]
        ].min(),
        "max": df[
            [
                "day_of_month",
                "week_of_year",
                "quarter",
                "is_weekend",
                "is_event",
            ]
        ].max(),
        "missing": df[
            [
                "day_of_month",
                "week_of_year",
                "quarter",
                "is_weekend",
                "is_event",
            ]
        ].isna().sum(),
    }
)

calendar_feature_summary

,min,max,missing
day_of_month,1,31,0
week_of_year,1,53,0
quarter,1,4,0
is_weekend,0,1,0
is_event,0,1,0


## 2. Lagged Demand Features

Historical demand is expected to provide some of the strongest predictive
signals for future product sales.

Lag features represent sales observed for the same product on earlier dates.
They allow forecasting models to learn short-term persistence, weekly
seasonality, and longer recurring demand patterns.

The initial lag horizons are:

- 1 day for recent demand;
- 7 days for weekly recurrence;
- 14 days for a second weekly horizon;
- 28 days for a four-week demand reference.

All lag features use historical sales only. The current day's sales value is
never included in its own predictors.

In [8]:
# ---------------------------------------------------------
# Sort each product time series chronologically
# ---------------------------------------------------------
# pandas shift() operates according to row order rather than
# automatically understanding chronological time.
#
# Sorting is therefore essential before constructing lagged
# or rolling demand features.

df = (
    df.sort_values(
        [
            "item_id",
            "date",
        ]
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# Verify chronological ordering within products
# ---------------------------------------------------------
# A negative date difference would indicate that at least one
# product's observations are not correctly ordered.

date_differences = (
    df.groupby(
        "item_id",
        observed=True
    )["date"]
    .diff()
)

out_of_order_rows = (
    date_differences
    .lt(pd.Timedelta(0))
    .sum()
)

print(
    "Out-of-order observations:",
    f"{out_of_order_rows:,}"
)

Out-of-order observations: 0


In [9]:
# ---------------------------------------------------------
# Check the spacing between consecutive observations
# ---------------------------------------------------------
# shift(7) means "seven rows earlier", not automatically
# "seven calendar days earlier".
#
# Therefore, we verify that consecutive observations for each
# product are exactly one calendar day apart. If this holds,
# row-based lags can safely be interpreted as day-based lags.

date_gaps = (
    df.groupby(
        "item_id",
        observed=True
    )["date"]
    .diff()
)


# ---------------------------------------------------------
# Examine non-standard gaps
# ---------------------------------------------------------
# The first observation for each product naturally has NaT
# because there is no previous observation. Those rows are
# excluded from this check.

valid_date_gaps = date_gaps.dropna()

non_daily_gaps = (
    valid_date_gaps
    .ne(pd.Timedelta(days=1))
    .sum()
)

print(
    "Consecutive date comparisons:",
    f"{len(valid_date_gaps):,}"
)

print(
    "Non-daily gaps:",
    f"{non_daily_gaps:,}"
)

print(
    "Minimum date gap:",
    valid_date_gaps.min()
)

print(
    "Maximum date gap:",
    valid_date_gaps.max()
)

Consecutive date comparisons: 5,915,060
Non-daily gaps: 0
Minimum date gap: 1 days 00:00:00
Maximum date gap: 1 days 00:00:00


In [10]:
# ---------------------------------------------------------
# Define demand lag horizons
# ---------------------------------------------------------
# These horizons capture different forms of temporal demand
# information:
#
# 1 day  -> immediate recent demand
# 7 days -> same point in the previous weekly cycle
# 14 days -> two-week historical reference
# 28 days -> four-week historical reference

lag_days = [
    1,
    7,
    14,
    28,
]


# ---------------------------------------------------------
# Create lagged sales features within each product
# ---------------------------------------------------------
# shift() ensures that only historical sales values are used.
# The current day's sales value therefore cannot enter its own
# lagged predictors.
#
# Because the product series were previously sorted and daily
# continuity was verified, each shift corresponds directly to
# the specified number of calendar days.

for lag in lag_days:
    df[f"sales_lag_{lag}"] = (
        df.groupby(
            "item_id",
            observed=True
        )["sales"]
        .shift(lag)
    )

In [11]:
# ---------------------------------------------------------
# Validate lag feature construction
# ---------------------------------------------------------
# Missing values are expected at the beginning of each product
# series because insufficient historical observations exist.
#
# With 3,049 products, we expect approximately:
#
# lag 1  ->  3,049 missing rows
# lag 7  -> 21,343 missing rows
# lag 14 -> 42,686 missing rows
# lag 28 -> 85,372 missing rows

lag_columns = [
    f"sales_lag_{lag}"
    for lag in lag_days
]

lag_validation = pd.DataFrame(
    {
        "missing_rows": (
            df[lag_columns]
            .isna()
            .sum()
        ),
        "missing_pct": (
            df[lag_columns]
            .isna()
            .mean()
            * 100
        ),
    }
)

lag_validation

,missing_rows,missing_pct
sales_lag_1,3049,0.051520
sales_lag_7,21343,0.360639
sales_lag_14,42686,0.721278
sales_lag_28,85372,1.442555


In [12]:
# ---------------------------------------------------------
# Select one product for manual lag verification
# ---------------------------------------------------------
# The first product is sufficient for this structural check.
# We inspect observations only after the first 28 days so that
# all four lag features are populated.

sample_item = df["item_id"].iloc[0]

sample_lag_check = (
    df.loc[
        df["item_id"].eq(sample_item),
        [
            "date",
            "sales",
            "sales_lag_1",
            "sales_lag_7",
            "sales_lag_14",
            "sales_lag_28",
        ],
    ]
    .iloc[28:38]
    .copy()
)

print("Sample product:", sample_item)

sample_lag_check

Sample product: FOODS_1_001


,date,sales,sales_lag_1,sales_lag_7,sales_lag_14,sales_lag_28
28,2011-02-26,2,4.0,1.0,3.0,3.0
29,2011-02-27,2,2.0,2.0,0.0,0.0
30,2011-02-28,0,2.0,0.0,2.0,0.0
31,2011-03-01,2,0.0,2.0,1.0,1.0
32,2011-03-02,1,2.0,2.0,2.0,4.0
33,2011-03-03,7,1.0,2.0,0.0,2.0
34,2011-03-04,1,7.0,4.0,2.0,0.0
35,2011-03-05,2,1.0,2.0,1.0,2.0
36,2011-03-06,3,2.0,2.0,2.0,0.0
37,2011-03-07,0,3.0,0.0,0.0,0.0


In [13]:
# ---------------------------------------------------------
# Programmatically verify lag alignment
# ---------------------------------------------------------
# We independently reconstruct the expected lagged dates for
# the selected product and compare their sales values with the
# generated lag columns.
#
# Every comparison should return True.

sample_history = (
    df.loc[
        df["item_id"].eq(sample_item),
        [
            "date",
            "sales",
        ],
    ]
    .set_index("date")["sales"]
)


lag_alignment_check = sample_lag_check[
    [
        "date",
        "sales",
    ]
].copy()


for lag in lag_days:

    # Look up the sales value exactly `lag` calendar days
    # before each observation date.
    lag_alignment_check[f"lag_{lag}_correct"] = [
        sample_history.get(
            date - pd.Timedelta(days=lag),
            np.nan
        )
        == lag_value

        for date, lag_value in zip(
            sample_lag_check["date"],
            sample_lag_check[f"sales_lag_{lag}"],
        )
    ]


lag_alignment_check

,date,sales,lag_1_correct,lag_7_correct,lag_14_correct,lag_28_correct
28,2011-02-26,2,True,True,True,True
29,2011-02-27,2,True,True,True,True
30,2011-02-28,0,True,True,True,True
31,2011-03-01,2,True,True,True,True
32,2011-03-02,1,True,True,True,True
33,2011-03-03,7,True,True,True,True
34,2011-03-04,1,True,True,True,True
35,2011-03-05,2,True,True,True,True
36,2011-03-06,3,True,True,True,True
37,2011-03-07,0,True,True,True,True


## 3. Rolling Demand Features

Lag features capture demand at specific historical points, while rolling
features summarize demand across recent historical windows.

Rolling statistics can help the forecasting model distinguish between
temporary fluctuations and broader changes in a product's recent demand
level.

Two initial windows are used:

- 7 days to represent recent weekly demand;
- 28 days to represent a broader four-week demand level.

To prevent target leakage, the sales series is shifted by one day before
calculating each rolling statistic. Therefore, the current day's sales value
is never included in its own rolling features.

In [14]:
# ---------------------------------------------------------
# Create leakage-safe rolling demand means
# ---------------------------------------------------------
# transform() preserves the original DataFrame index, making
# assignment back to df straightforward.
#
# Inside each product series:
#
#   shift(1)
#
# removes the current day's target before rolling() is applied.
#
# For example, rolling_mean_7 on day t summarizes sales from
# t-7 through t-1, never sales from day t itself.

df["rolling_mean_7"] = (
    df.groupby(
        "item_id",
        observed=True
    )["sales"]
    .transform(
        lambda x: (
            x.shift(1)
            .rolling(
                window=7,
                min_periods=7
            )
            .mean()
        )
    )
)


df["rolling_mean_28"] = (
    df.groupby(
        "item_id",
        observed=True
    )["sales"]
    .transform(
        lambda x: (
            x.shift(1)
            .rolling(
                window=28,
                min_periods=28
            )
            .mean()
        )
    )
)

In [15]:
# ---------------------------------------------------------
# Validate rolling-feature missing values
# ---------------------------------------------------------
# Because complete historical windows are required:
#
# rolling_mean_7  -> first 7 observations per product missing
# rolling_mean_28 -> first 28 observations per product missing
#
# With 3,049 products, these counts should match the
# corresponding 7-day and 28-day lag missing counts.

rolling_mean_columns = [
    "rolling_mean_7",
    "rolling_mean_28",
]

rolling_mean_validation = pd.DataFrame(
    {
        "missing_rows": (
            df[rolling_mean_columns]
            .isna()
            .sum()
        ),
        "missing_pct": (
            df[rolling_mean_columns]
            .isna()
            .mean()
            * 100
        ),
    }
)

rolling_mean_validation

,missing_rows,missing_pct
rolling_mean_7,21343,0.360639
rolling_mean_28,85372,1.442555


In [16]:
# ---------------------------------------------------------
# Select observations with complete rolling history
# ---------------------------------------------------------
# We begin after the first 28 days so that both the 7-day and
# 28-day rolling features are available for comparison.

sample_rolling_check = (
    df.loc[
        df["item_id"].eq(sample_item),
        [
            "date",
            "sales",
            "rolling_mean_7",
            "rolling_mean_28",
        ],
    ]
    .iloc[28:38]
    .copy()
)


# ---------------------------------------------------------
# Independently calculate expected historical rolling means
# ---------------------------------------------------------
# For each target date:
#
# 7-day mean  = sales from t-7 through t-1
# 28-day mean = sales from t-28 through t-1
#
# The current day's sales value is deliberately excluded.

expected_mean_7 = []
expected_mean_28 = []

for date in sample_rolling_check["date"]:

    historical_7 = sample_history.loc[
        (sample_history.index >= date - pd.Timedelta(days=7))
        & (sample_history.index < date)
    ]

    historical_28 = sample_history.loc[
        (sample_history.index >= date - pd.Timedelta(days=28))
        & (sample_history.index < date)
    ]

    expected_mean_7.append(
        historical_7.mean()
    )

    expected_mean_28.append(
        historical_28.mean()
    )


sample_rolling_check["expected_mean_7"] = (
    expected_mean_7
)

sample_rolling_check["expected_mean_28"] = (
    expected_mean_28
)


# ---------------------------------------------------------
# Compare generated and independently calculated values
# ---------------------------------------------------------
# np.isclose() is used instead of exact equality because
# rolling means are floating-point values.

sample_rolling_check["mean_7_correct"] = (
    np.isclose(
        sample_rolling_check["rolling_mean_7"],
        sample_rolling_check["expected_mean_7"],
    )
)

sample_rolling_check["mean_28_correct"] = (
    np.isclose(
        sample_rolling_check["rolling_mean_28"],
        sample_rolling_check["expected_mean_28"],
    )
)


sample_rolling_check[
    [
        "date",
        "sales",
        "rolling_mean_7",
        "expected_mean_7",
        "mean_7_correct",
        "rolling_mean_28",
        "expected_mean_28",
        "mean_28_correct",
    ]
]

,date,sales,rolling_mean_7,expected_mean_7,mean_7_correct,rolling_mean_28,expected_mean_28,mean_28_correct
28,2011-02-26,2,1.857143,1.857143,True,1.392857,1.392857,True
29,2011-02-27,2,2.000000,2.000000,True,1.357143,1.357143,True
30,2011-02-28,0,2.000000,2.000000,True,1.428571,1.428571,True
31,2011-03-01,2,2.000000,2.000000,True,1.428571,1.428571,True
32,2011-03-02,1,2.000000,2.000000,True,1.464286,1.464286,True
33,2011-03-03,7,1.857143,1.857143,True,1.357143,1.357143,True
34,2011-03-04,1,2.571429,2.571429,True,1.535714,1.535714,True
35,2011-03-05,2,2.142857,2.142857,True,1.571429,1.571429,True
36,2011-03-06,3,2.142857,2.142857,True,1.571429,1.571429,True
37,2011-03-07,0,2.285714,2.285714,True,1.678571,1.678571,True


In [17]:
# ---------------------------------------------------------
# Create leakage-safe rolling demand variability features
# ---------------------------------------------------------
# Standard deviation measures how much recent demand fluctuates
# around its local mean.
#
# As with the rolling means, sales are shifted by one day
# before calculating the statistic so that the current target
# is never included in its own feature values.

df["rolling_std_7"] = (
    df.groupby(
        "item_id",
        observed=True
    )["sales"]
    .transform(
        lambda x: (
            x.shift(1)
            .rolling(
                window=7,
                min_periods=7
            )
            .std()
        )
    )
)


df["rolling_std_28"] = (
    df.groupby(
        "item_id",
        observed=True
    )["sales"]
    .transform(
        lambda x: (
            x.shift(1)
            .rolling(
                window=28,
                min_periods=28
            )
            .std()
        )
    )
)

In [18]:
# ---------------------------------------------------------
# Validate rolling variability features
# ---------------------------------------------------------
# With complete-window requirements, the missing-value pattern
# should match the corresponding rolling means.

rolling_std_columns = [
    "rolling_std_7",
    "rolling_std_28",
]

rolling_std_validation = pd.DataFrame(
    {
        "missing_rows": (
            df[rolling_std_columns]
            .isna()
            .sum()
        ),
        "missing_pct": (
            df[rolling_std_columns]
            .isna()
            .mean()
            * 100
        ),
        "min_value": (
            df[rolling_std_columns]
            .min()
        ),
        "max_value": (
            df[rolling_std_columns]
            .max()
        ),
    }
)

rolling_std_validation

,missing_rows,missing_pct,min_value,max_value
rolling_std_7,21343,0.360639,0.0,241.586226
rolling_std_28,85372,1.442555,0.0,131.727969


## 4. Price and Promotion Features

Exploratory analysis showed that selling prices are generally stable, but
products occasionally experience substantial temporary markdowns.

Raw percentage price changes can become unstable when the previous price is
close to zero. Price features are therefore designed to emphasize historical
price levels and product-relative pricing while avoiding unnecessary
sensitivity to near-zero denominators.

Feature construction also follows the same leakage-prevention principle used
for demand features. Historical observations must not use future price
information.

In [19]:
# ---------------------------------------------------------
# Create previous-day selling price
# ---------------------------------------------------------
# Because each product has a continuous daily time series,
# shift(1) represents the selling price recorded exactly one
# calendar day earlier.
#
# The shift is performed within each product so that price
# information never crosses product boundaries.

df["sell_price_lag_1"] = (
    df.groupby(
        "item_id",
        observed=True
    )["sell_price"]
    .shift(1)
)

In [20]:
# ---------------------------------------------------------
# Inspect previous-day price availability
# ---------------------------------------------------------
# Missing current prices correspond to unavailable product
# periods identified during data preparation.
#
# Here we examine how often an available product has no price
# recorded on the immediately preceding day. This can occur
# when a product first becomes available after an unavailable
# period.

price_lag_check = pd.DataFrame(
    {
        "current_available": df["sell_price"].notna(),
        "previous_day_price_available": (
            df["sell_price_lag_1"].notna()
        ),
    }
)


available_rows = (
    price_lag_check["current_available"]
)

available_without_previous_price = (
    available_rows
    & ~price_lag_check["previous_day_price_available"]
)


print(
    "Available item-day observations:",
    f"{available_rows.sum():,}"
)

print(
    "Available observations without previous-day price:",
    f"{available_without_previous_price.sum():,}"
)

print(
    "Share of available observations without previous-day price:",
    f"{available_without_previous_price.sum() / available_rows.sum() * 100:.2f}%"
)

Available item-day observations: 4,788,267
Available observations without previous-day price: 3,049
Share of available observations without previous-day price: 0.06%


In [21]:
# ---------------------------------------------------------
# Investigate available observations with no previous-day price
# ---------------------------------------------------------
# There are exactly 3,049 such observations, matching the
# number of products.
#
# We verify whether each product contributes exactly one such
# observation and whether it corresponds to that product's
# first available date.

missing_previous_price = (
    df.loc[
        df["sell_price"].notna()
        & df["sell_price_lag_1"].isna(),
        [
            "item_id",
            "date",
            "sell_price",
        ],
    ]
    .copy()
)


# ---------------------------------------------------------
# Count affected observations per product
# ---------------------------------------------------------

missing_per_product = (
    missing_previous_price
    .groupby(
        "item_id",
        observed=True
    )
    .size()
)


print(
    "Products represented:",
    f"{missing_previous_price['item_id'].nunique():,}"
)

print(
    "Minimum missing observations per product:",
    missing_per_product.min()
)

print(
    "Maximum missing observations per product:",
    missing_per_product.max()
)


# ---------------------------------------------------------
# Compare with each product's first available date
# ---------------------------------------------------------
# If every comparison is True, the missing lagged price is
# simply a structural consequence of a product entering its
# first observed available period.

first_available_date = (
    df.loc[
        df["sell_price"].notna()
    ]
    .groupby(
        "item_id",
        observed=True
    )["date"]
    .min()
    .rename("first_available_date")
    .reset_index()
)


missing_previous_price = (
    missing_previous_price.merge(
        first_available_date,
        on="item_id",
        how="left",
        validate="many_to_one",
    )
)


missing_previous_price["is_first_available_date"] = (
    missing_previous_price["date"]
    .eq(
        missing_previous_price["first_available_date"]
    )
)


print(
    "Observations occurring on first available date:",
    f"{missing_previous_price['is_first_available_date'].sum():,}"
)

print(
    "All missing previous prices are first availability:",
    missing_previous_price["is_first_available_date"].all()
)

Products represented: 3,049
Minimum missing observations per product: 1
Maximum missing observations per product: 1
Observations occurring on first available date: 3,049
All missing previous prices are first availability: True


In [22]:
# ---------------------------------------------------------
# Create previous-day price-change indicator
# ---------------------------------------------------------
# A price change can only be evaluated when both today's
# selling price and yesterday's selling price are available.
#
# The first available observation for each product has no
# historical price for comparison, so it remains missing
# rather than being classified as either changed or unchanged.

comparable_price = (
    df["sell_price"].notna()
    & df["sell_price_lag_1"].notna()
)


df["price_changed"] = np.where(
    comparable_price,
    ~np.isclose(
        df["sell_price"],
        df["sell_price_lag_1"],
    ),
    np.nan,
)

In [23]:
# ---------------------------------------------------------
# Summarize daily price-change observations
# ---------------------------------------------------------
# M5 selling prices are weekly, so most consecutive available
# days should naturally have identical prices.
#
# The purpose of this feature is to identify the relatively
# uncommon dates on which a new price becomes active.

price_change_summary = (
    df.loc[
        comparable_price,
        "price_changed",
    ]
    .value_counts()
    .rename_axis("price_changed")
    .reset_index(name="observations")
)


price_change_summary["percentage"] = (
    price_change_summary["observations"]
    / price_change_summary["observations"].sum()
    * 100
)

price_change_summary

,price_changed,observations,percentage
0,0.0,4778898,99.867927
1,1.0,6320,0.132073


## Forecast-Horizon Feature Availability

The forecasting objective uses a 28-day prediction horizon, consistent with
the M5 forecasting setup.

This creates an additional constraint beyond ordinary target leakage:
features must be available at the forecast origin for every day in the
28-day prediction window.

For example, a one-day sales lag is valid for one-step-ahead forecasting,
but it would not be known for later days in a 28-day forecast window unless
predictions were generated recursively.

To keep the initial modeling framework non-recursive and leakage-safe,
historical demand features used by the final model will therefore be based
on information at least 28 days before each target date.

Examples include:

- sales lagged by 28, 35, 42, and 56 days;
- rolling demand statistics calculated after a 28-day shift.

The previously constructed 1-, 7-, and 14-day lags remain useful for
understanding feature construction, but they will not be used as predictors
in the initial 28-day direct forecasting model.

In [24]:
# ---------------------------------------------------------
# Create horizon-safe demand lag features
# ---------------------------------------------------------
# The forecasting horizon is 28 days.
#
# Therefore, the final non-recursive forecasting model should
# only use historical sales that would already be known at the
# beginning of the entire 28-day prediction window.
#
# Using multiples of seven also preserves weekday alignment,
# which is valuable because EDA showed strong weekly seasonality.

forecast_horizon = 28

safe_lag_days = [
    28,  # Four weeks earlier
    35,  # Five weeks earlier
    42,  # Six weeks earlier
    56,  # Eight weeks earlier
]


for lag in safe_lag_days:

    column_name = f"sales_lag_{lag}"

    # sales_lag_28 already exists from the earlier feature
    # engineering step. Recreating it is harmless, but we avoid
    # unnecessary work on the 5.9M-row dataset.
    if column_name not in df.columns:

        df[column_name] = (
            df.groupby(
                "item_id",
                observed=True
            )["sales"]
            .shift(lag)
        )

In [25]:
# ---------------------------------------------------------
# Validate horizon-safe lag features
# ---------------------------------------------------------
# Because every product has a continuous daily history,
# a lag of N days should create exactly:
#
#     number_of_products × N
#
# missing observations at the beginning of the series.

n_products = df["item_id"].nunique()

safe_lag_validation = []

for lag in safe_lag_days:

    column_name = f"sales_lag_{lag}"

    actual_missing = (
        df[column_name]
        .isna()
        .sum()
    )

    expected_missing = (
        n_products * lag
    )

    safe_lag_validation.append(
        {
            "feature": column_name,
            "actual_missing": actual_missing,
            "expected_missing": expected_missing,
            "matches_expected": (
                actual_missing == expected_missing
            ),
        }
    )


safe_lag_validation = pd.DataFrame(
    safe_lag_validation
)

safe_lag_validation

,feature,actual_missing,expected_missing,matches_expected
0,sales_lag_28,85372,85372,True
1,sales_lag_35,106715,106715,True
2,sales_lag_42,128058,128058,True
3,sales_lag_56,170744,170744,True


In [26]:
# ---------------------------------------------------------
# Create 28-day-horizon-safe rolling demand features
# ---------------------------------------------------------
# The sales series is shifted by the full forecast horizon
# BEFORE calculating rolling statistics.
#
# This ensures that every observation used in these features
# would already be known when the 28-day forecast is created.
#
# Example for target day t:
#
# rolling_mean_7_lag28
#     -> sales from t-34 through t-28
#
# rolling_mean_28_lag28
#     -> sales from t-55 through t-28

historical_sales_at_origin = (
    df.groupby(
        "item_id",
        observed=True
    )["sales"]
    .shift(forecast_horizon)
)


# ---------------------------------------------------------
# 7-day historical demand level
# ---------------------------------------------------------

df["rolling_mean_7_lag28"] = (
    historical_sales_at_origin
    .groupby(
        df["item_id"],
        observed=True
    )
    .transform(
        lambda x: x.rolling(
            window=7,
            min_periods=7
        ).mean()
    )
)


# ---------------------------------------------------------
# 28-day historical demand level
# ---------------------------------------------------------

df["rolling_mean_28_lag28"] = (
    historical_sales_at_origin
    .groupby(
        df["item_id"],
        observed=True
    )
    .transform(
        lambda x: x.rolling(
            window=28,
            min_periods=28
        ).mean()
    )
)

In [27]:
# ---------------------------------------------------------
# Create horizon-safe demand variability features
# ---------------------------------------------------------
# These measure how volatile demand was during historical
# periods that were fully observable at forecast time.

df["rolling_std_7_lag28"] = (
    historical_sales_at_origin
    .groupby(
        df["item_id"],
        observed=True
    )
    .transform(
        lambda x: x.rolling(
            window=7,
            min_periods=7
        ).std()
    )
)


df["rolling_std_28_lag28"] = (
    historical_sales_at_origin
    .groupby(
        df["item_id"],
        observed=True
    )
    .transform(
        lambda x: x.rolling(
            window=28,
            min_periods=28
        ).std()
    )
)

In [28]:
# ---------------------------------------------------------
# Validate horizon-safe rolling features
# ---------------------------------------------------------

horizon_safe_rolling = {
    "rolling_mean_7_lag28": 34,
    "rolling_std_7_lag28": 34,
    "rolling_mean_28_lag28": 55,
    "rolling_std_28_lag28": 55,
}

rolling_validation = []

for feature, expected_initial_missing_days in (
    horizon_safe_rolling.items()
):

    actual_missing = (
        df[feature]
        .isna()
        .sum()
    )

    expected_missing = (
        n_products
        * expected_initial_missing_days
    )

    rolling_validation.append(
        {
            "feature": feature,
            "actual_missing": actual_missing,
            "expected_missing": expected_missing,
            "matches_expected": (
                actual_missing == expected_missing
            ),
        }
    )


rolling_validation = pd.DataFrame(
    rolling_validation
)

rolling_validation

,feature,actual_missing,expected_missing,matches_expected
0,rolling_mean_7_lag28,103666,103666,True
1,rolling_std_7_lag28,103666,103666,True
2,rolling_mean_28_lag28,167695,167695,True
3,rolling_std_28_lag28,167695,167695,True


In [29]:
# ---------------------------------------------------------
# Remove demand features that are unsafe for the final
# non-recursive 28-day forecasting design
# ---------------------------------------------------------
# These features were useful for validating our feature
# engineering logic, but they rely on information that would
# not necessarily be available for every day of a 28-day
# forecast generated at a single forecast origin.
#
# sales_lag_28 is deliberately retained because it is
# horizon-safe.

unsafe_demand_features = [
    "sales_lag_1",
    "sales_lag_7",
    "sales_lag_14",
    "rolling_mean_7",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_28",
]


df = df.drop(
    columns=unsafe_demand_features
)

In [30]:
# ---------------------------------------------------------
# Verify the final historical-demand feature block
# ---------------------------------------------------------
# Only horizon-safe demand predictors should remain.

demand_feature_columns = [
    column
    for column in df.columns
    if (
        column.startswith("sales_lag_")
        or column.startswith("rolling_")
    )
]

demand_feature_columns

['sales_lag_28',
 'sales_lag_35',
 'sales_lag_42',
 'sales_lag_56',
 'rolling_mean_7_lag28',
 'rolling_mean_28_lag28',
 'rolling_std_7_lag28',
 'rolling_std_28_lag28']

In [31]:
# ---------------------------------------------------------
# Create horizon-safe historical price feature
# ---------------------------------------------------------
# sell_price_lag_28 represents the product's selling price
# exactly 28 days before the target date.
#
# Unlike sell_price_lag_1, this historical price is available
# at the forecast origin for every day in the 28-day horizon.

df["sell_price_lag_28"] = (
    df.groupby(
        "item_id",
        observed=True
    )["sell_price"]
    .shift(forecast_horizon)
)

In [32]:
# ---------------------------------------------------------
# Validate availability of the 28-day historical price
# ---------------------------------------------------------
# Some lagged prices can legitimately be missing because the
# product may not have been available 28 days earlier.
#
# We therefore quantify the pattern rather than automatically
# imputing missing prices.

current_price_available = (
    df["sell_price"].notna()
)

lag28_price_missing = (
    df["sell_price_lag_28"].isna()
)

available_with_lag28 = (
    current_price_available
    & ~lag28_price_missing
)

available_without_lag28 = (
    current_price_available
    & lag28_price_missing
)


print(
    "Current-price available observations:",
    f"{current_price_available.sum():,}"
)

print(
    "Available observations with 28-day historical price:",
    f"{available_with_lag28.sum():,}"
)

print(
    "Available observations without 28-day historical price:",
    f"{available_without_lag28.sum():,}"
)

print(
    "Share without 28-day historical price:",
    (
        f"{available_without_lag28.sum() / current_price_available.sum() * 100:.2f}%"
    )
)

Current-price available observations: 4,788,267
Available observations with 28-day historical price: 4,702,895
Available observations without 28-day historical price: 85,372
Share without 28-day historical price: 1.78%


In [33]:
# ---------------------------------------------------------
# Create horizon-safe absolute price difference
# ---------------------------------------------------------
# This feature compares the target day's known selling price
# with the product's price 28 days earlier.
#
# A positive value means the current price is higher than it
# was four weeks earlier.
#
# A negative value means the current price is lower.
#
# A value of zero means the price is unchanged.
#
# We deliberately start with an absolute difference rather
# than percentage change because EDA showed that percentage
# changes can become extremely large when the historical
# price is close to zero.

df["price_diff_28"] = (
    df["sell_price"]
    - df["sell_price_lag_28"]
)

In [34]:
# ---------------------------------------------------------
# Inspect the 28-day price-difference feature
# ---------------------------------------------------------
# Missing values are expected whenever either the current
# price or the 28-day historical price is unavailable.

price_diff_summary = (
    df["price_diff_28"]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
)

print(price_diff_summary)

print(
    "\nMissing price_diff_28 values:",
    f"{df['price_diff_28'].isna().sum():,}"
)

print(
    "Non-zero price differences:",
    f"{(df['price_diff_28'].fillna(0).abs() > 1e-8).sum():,}"
)

count    4.702895e+06
mean     1.734064e-03
std      1.267679e-01
min     -1.597000e+01
1%      -1.000000e-01
5%       0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
95%      0.000000e+00
99%      2.000000e-01
max      1.597000e+01
Name: price_diff_28, dtype: float64

Missing price_diff_28 values: 1,215,214
Non-zero price differences: 161,276


In [35]:
# ---------------------------------------------------------
# Create horizon-safe relative price feature
# ---------------------------------------------------------
# The ratio measures today's known/planned selling price
# relative to the price 28 days earlier.
#
# Examples:
#   1.00 -> unchanged
#   0.90 -> current price is 10% lower
#   1.10 -> current price is 10% higher
#
# Extremely small historical prices can produce unstable
# ratios. We therefore require the historical price to be
# meaningfully above zero before calculating the ratio.

valid_price_ratio = (
    df["sell_price"].notna()
    & df["sell_price_lag_28"].notna()
    & (df["sell_price_lag_28"] > 0.01)
)


df["price_ratio_28"] = np.where(
    valid_price_ratio,
    (
        df["sell_price"]
        / df["sell_price_lag_28"]
    ),
    np.nan,
)

In [36]:
# ---------------------------------------------------------
# Validate the relative price feature
# ---------------------------------------------------------
# Most observations should be close to 1 because the EDA
# showed that product prices change relatively infrequently.
#
# We inspect the tails carefully because temporary markdowns
# can create unusually large or small ratios.

price_ratio_summary = (
    df["price_ratio_28"]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
)

print(price_ratio_summary)

print(
    "\nMissing price_ratio_28 values:",
    f"{df['price_ratio_28'].isna().sum():,}"
)

print(
    "Ratios below 0.90:",
    f"{(df['price_ratio_28'] < 0.90).sum():,}"
)

print(
    "Ratios above 1.10:",
    f"{(df['price_ratio_28'] > 1.10).sum():,}"
)

count    4.702888e+06
mean     1.000974e+00
std      7.847156e-02
min      4.048583e-03
1%       9.664430e-01
5%       1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
95%      1.000000e+00
99%      1.071942e+00
max      5.760000e+01
Name: price_ratio_28, dtype: float64

Missing price_ratio_28 values: 1,215,221
Ratios below 0.90: 20,934
Ratios above 1.10: 30,992


In [37]:
# ---------------------------------------------------------
# Create horizon-safe log relative price feature
# ---------------------------------------------------------
# Taking the logarithm of the price ratio provides a more
# stable representation of relative price movements.
#
# Interpretation:
#   0     -> unchanged price
#   < 0   -> current price is lower than 28 days earlier
#   > 0   -> current price is higher than 28 days earlier
#
# The log transformation also makes proportional increases
# and decreases more symmetric and compresses extreme ratios
# caused by temporary near-zero markdown prices.
#
# price_ratio_28 already contains NaN where a reliable ratio
# could not be calculated, and log(NaN) remains NaN.

df["log_price_ratio_28"] = (
    np.log(
        df["price_ratio_28"]
    )
)

In [38]:
# ---------------------------------------------------------
# Validate the log-relative price feature
# ---------------------------------------------------------

log_price_summary = (
    df["log_price_ratio_28"]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
)

print(log_price_summary)

print(
    "\nMissing log_price_ratio_28 values:",
    f"{df['log_price_ratio_28'].isna().sum():,}"
)

print(
    "Infinite values:",
    f"{np.isinf(df['log_price_ratio_28']).sum():,}"
)

count    4.702888e+06
mean     4.379330e-04
std      3.047499e-02
min     -5.509388e+00
1%      -3.413301e-02
5%       0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
95%      0.000000e+00
99%      6.947237e-02
max      4.053523e+00
Name: log_price_ratio_28, dtype: float64

Missing log_price_ratio_28 values: 1,215,221
Infinite values: 0


In [39]:
# ---------------------------------------------------------
# Remove diagnostic and redundant price features
# ---------------------------------------------------------
# sell_price_lag_1 and price_changed were useful for
# understanding the structure of the M5 price data, but they
# are not appropriate historical predictors for our fixed
# 28-day forecasting horizon.
#
# price_ratio_28 is removed because log_price_ratio_28
# represents the same proportional price information while
# handling extreme ratios more gracefully.

price_features_to_remove = [
    "sell_price_lag_1",
    "price_changed",
    "price_ratio_28",
]


df = df.drop(
    columns=price_features_to_remove
)

In [40]:
# ---------------------------------------------------------
# Verify final price-related model features
# ---------------------------------------------------------

final_price_features = [
    "sell_price",
    "sell_price_lag_28",
    "price_diff_28",
    "log_price_ratio_28",
    "is_available",
]


price_feature_validation = pd.DataFrame(
    {
        "dtype": df[final_price_features].dtypes.astype(str),
        "missing_rows": df[final_price_features].isna().sum(),
        "missing_pct": (
            df[final_price_features].isna().mean()
            * 100
        ),
    }
)

price_feature_validation

,dtype,missing_rows,missing_pct
sell_price,float64,1129842,19.091267
sell_price_lag_28,float64,1215214,20.533823
price_diff_28,float64,1215214,20.533823
log_price_ratio_28,float64,1215221,20.533941
is_available,uint8,0,0.000000


In [41]:
# ---------------------------------------------------------
# Audit the complete feature-engineering dataset
# ---------------------------------------------------------
# Before creating chronological train/validation/test splits,
# we inspect every column currently present in the dataset.
#
# This audit helps us distinguish:
#   1. target and identifiers;
#   2. categorical product information;
#   3. calendar/event information;
#   4. horizon-safe demand features;
#   5. price and availability features.
#
# We also inspect data types and missingness before deciding
# which columns will actually enter the forecasting model.

feature_audit = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing_rows": df.isna().sum(),
        "missing_pct": (
            df.isna().mean()
            * 100
        ),
        "unique_values": df.nunique(
            dropna=True
        ),
    }
)


print(
    "Dataset shape:",
    df.shape
)

print(
    "Number of columns:",
    df.shape[1]
)

feature_audit

Dataset shape: (5918109, 37)
Number of columns: 37


,dtype,missing_rows,missing_pct,unique_values
id,category,0,0.000000,3049
item_id,category,0,0.000000,3049
dept_id,category,0,0.000000,7
cat_id,category,0,0.000000,3
store_id,str,0,0.000000,1
state_id,category,0,0.000000,1
d,str,0,0.000000,1941
sales,uint16,0,0.000000,236
date,datetime64[us],0,0.000000,1941
wm_yr_wk,int64,0,0.000000,278


## 5. Modeling Feature Set

The feature-engineering dataset contains both modeling predictors and
structural columns used for identification, validation, and time-based
splitting.

For the initial forecasting model:

- `sales` is the prediction target;
- `date` is retained for chronological splitting and evaluation;
- `item_id` identifies individual products and is retained as a categorical
  predictor;
- product hierarchy variables (`dept_id`, `cat_id`) are retained;
- calendar, event, SNAP, demand-history, price, and availability variables
  are retained as predictors;
- constant single-store variables are excluded because they contain no
  variation;
- redundant identifiers such as `id` and `d` are excluded from the model.

The feature set is defined explicitly rather than automatically selecting
all columns. This reduces the risk of accidentally introducing identifiers,
diagnostic variables, or future leakage into the forecasting model.

In [42]:
# ---------------------------------------------------------
# Define target and structural columns
# ---------------------------------------------------------
# These columns are required for identification, chronological
# splitting, or evaluation, but are not model predictors.

target_column = "sales"

split_column = "date"

excluded_columns = [
    "id",        # Redundant with item_id in this single-store dataset
    "d",         # M5 sequential day identifier; date is used instead
    "store_id",  # Constant because the project currently models CA_1 only
    "state_id",  # Constant because CA_1 belongs to a single state
]


# ---------------------------------------------------------
# Define categorical predictors
# ---------------------------------------------------------
# These represent product hierarchy and calendar/event
# categories rather than continuous numeric quantities.

categorical_features = [
    "item_id",
    "dept_id",
    "cat_id",
    "weekday",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
]


# ---------------------------------------------------------
# Define numeric predictors
# ---------------------------------------------------------
# Demand-history features are all horizon-safe: they use
# information at least 28 days before the target date.
#
# Current sell_price and availability are treated as known
# target-period information, consistent with the M5 setup.

numeric_features = [
    # Calendar features
    "wday",
    "month",
    "year",
    "wm_yr_wk",
    "snap_CA",
    "day_of_month",
    "week_of_year",
    "quarter",
    "is_weekend",
    "is_event",

    # Horizon-safe historical demand features
    "sales_lag_28",
    "sales_lag_35",
    "sales_lag_42",
    "sales_lag_56",
    "rolling_mean_7_lag28",
    "rolling_mean_28_lag28",
    "rolling_std_7_lag28",
    "rolling_std_28_lag28",

    # Price and availability features
    "sell_price",
    "sell_price_lag_28",
    "price_diff_28",
    "log_price_ratio_28",
    "is_available",
]


# ---------------------------------------------------------
# Construct the final predictor list
# ---------------------------------------------------------

model_features = (
    categorical_features
    + numeric_features
)


print("Categorical features:", len(categorical_features))
print("Numeric features:", len(numeric_features))
print("Total model features:", len(model_features))

Categorical features: 8
Numeric features: 23
Total model features: 31


In [43]:
# ---------------------------------------------------------
# Validate the feature contract
# ---------------------------------------------------------
# This catches accidental duplicate names, missing columns,
# or leakage of the target/date into the predictor list.

missing_model_features = [
    column
    for column in model_features
    if column not in df.columns
]

duplicate_model_features = [
    column
    for column in set(model_features)
    if model_features.count(column) > 1
]


print(
    "Missing model features:",
    missing_model_features
)

print(
    "Duplicate model features:",
    duplicate_model_features
)

print(
    "Target included in predictors:",
    target_column in model_features
)

print(
    "Date included in predictors:",
    split_column in model_features
)

Missing model features: []
Duplicate model features: []
Target included in predictors: False
Date included in predictors: False


## 6. Chronological Train, Validation, and Test Split

Forecasting models must be evaluated on future observations rather than
randomly sampled records.

The dataset is therefore divided chronologically:

- **Training set:** all observations before the final 56 days;
- **Validation set:** the first 28 days of the final 56-day period;
- **Test set:** the final 28 days.

The validation period is used for model development, feature decisions, and
model comparison.

The final test period remains untouched during model development and is used
only for the final out-of-sample evaluation.

This design produces two consecutive 28-day forecast windows and aligns the
evaluation framework with the project's 28-day forecasting horizon.

In [44]:
# ---------------------------------------------------------
# Determine chronological split boundaries
# ---------------------------------------------------------
# We reserve the final two 28-day periods:
#
#   validation -> 28 days
#   test       -> 28 days
#
# The dates are derived directly from the dataset so that the
# split logic remains reproducible and does not depend on
# manually entered calendar dates.

forecast_horizon = 28

unique_dates = (
    df["date"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)


# The test period contains the final 28 unique dates.
test_dates = unique_dates.iloc[
    -forecast_horizon:
]


# The validation period contains the preceding 28 dates.
validation_dates = unique_dates.iloc[
    -(2 * forecast_horizon):-forecast_horizon
]


# Everything before validation belongs to training.
training_dates = unique_dates.iloc[
    :-(2 * forecast_horizon)
]


print(
    "Training:",
    training_dates.min().date(),
    "to",
    training_dates.max().date(),
    f"({len(training_dates):,} days)"
)

print(
    "Validation:",
    validation_dates.min().date(),
    "to",
    validation_dates.max().date(),
    f"({len(validation_dates):,} days)"
)

print(
    "Test:",
    test_dates.min().date(),
    "to",
    test_dates.max().date(),
    f"({len(test_dates):,} days)"
)

Training: 2011-01-29 to 2016-03-27 (1,885 days)
Validation: 2016-03-28 to 2016-04-24 (28 days)
Test: 2016-04-25 to 2016-05-22 (28 days)


In [45]:
# ---------------------------------------------------------
# Validate chronological split integrity
# ---------------------------------------------------------
# These checks ensure that:
#
#   training ends before validation starts;
#   validation ends before test starts;
#   validation contains exactly 28 days;
#   test contains exactly 28 days.

split_validation = {
    "training_before_validation": (
        training_dates.max()
        < validation_dates.min()
    ),

    "validation_before_test": (
        validation_dates.max()
        < test_dates.min()
    ),

    "validation_is_28_days": (
        len(validation_dates)
        == forecast_horizon
    ),

    "test_is_28_days": (
        len(test_dates)
        == forecast_horizon
    ),

    "total_dates_preserved": (
        len(training_dates)
        + len(validation_dates)
        + len(test_dates)
        == len(unique_dates)
    ),
}


split_validation

{'training_before_validation': True,
 'validation_before_test': True,
 'validation_is_28_days': True,
 'test_is_28_days': True,
 'total_dates_preserved': True}

In [46]:
# ---------------------------------------------------------
# Create chronological train, validation, and test datasets
# ---------------------------------------------------------
# Boolean date masks are used rather than random sampling.
# This preserves the temporal ordering required for a
# forecasting problem.
#
# The .copy() calls ensure that each split is an independent
# DataFrame and avoid chained-assignment issues later.

train_df = (
    df.loc[
        df["date"].isin(training_dates)
    ]
    .copy()
)

validation_df = (
    df.loc[
        df["date"].isin(validation_dates)
    ]
    .copy()
)

test_df = (
    df.loc[
        df["date"].isin(test_dates)
    ]
    .copy()
)


# ---------------------------------------------------------
# Summarize the resulting splits
# ---------------------------------------------------------

split_summary = pd.DataFrame(
    {
        "split": [
            "train",
            "validation",
            "test",
        ],
        "rows": [
            len(train_df),
            len(validation_df),
            len(test_df),
        ],
        "days": [
            train_df["date"].nunique(),
            validation_df["date"].nunique(),
            test_df["date"].nunique(),
        ],
        "products": [
            train_df["item_id"].nunique(),
            validation_df["item_id"].nunique(),
            test_df["item_id"].nunique(),
        ],
        "start_date": [
            train_df["date"].min(),
            validation_df["date"].min(),
            test_df["date"].min(),
        ],
        "end_date": [
            train_df["date"].max(),
            validation_df["date"].max(),
            test_df["date"].max(),
        ],
    }
)

split_summary

,split,rows,days,products,start_date,end_date
0,train,5747365,1885,3049,2011-01-29,2016-03-27
1,validation,85372,28,3049,2016-03-28,2016-04-24
2,test,85372,28,3049,2016-04-25,2016-05-22


In [47]:
# ---------------------------------------------------------
# Validate row-level split integrity
# ---------------------------------------------------------
# Every original row should belong to exactly one split.
# No rows should be lost or duplicated during partitioning.

total_split_rows = (
    len(train_df)
    + len(validation_df)
    + len(test_df)
)


print(
    "Original dataset rows:",
    f"{len(df):,}"
)

print(
    "Rows across all splits:",
    f"{total_split_rows:,}"
)

print(
    "All rows preserved:",
    total_split_rows == len(df)
)

Original dataset rows: 5,918,109
Rows across all splits: 5,918,109
All rows preserved: True


In [48]:
# ---------------------------------------------------------
# Audit model-feature missingness by dataset split
# ---------------------------------------------------------
# Missing values can arise for different reasons:
#
# 1. Historical demand features:
#    Missing only near the beginning of a product's history
#    because sufficient lag/rolling observations do not exist.
#
# 2. Price features:
#    Missingness can represent genuine product unavailability.
#
# We inspect each split separately before deciding how these
# cases should be handled by the forecasting model.

missing_by_split = pd.DataFrame(
    {
        "train_missing": (
            train_df[model_features]
            .isna()
            .sum()
        ),

        "validation_missing": (
            validation_df[model_features]
            .isna()
            .sum()
        ),

        "test_missing": (
            test_df[model_features]
            .isna()
            .sum()
        ),
    }
)


# Keep only features that contain at least one missing value
# in any of the three datasets.

missing_by_split = (
    missing_by_split.loc[
        missing_by_split.sum(axis=1) > 0
    ]
)

missing_by_split

,train_missing,validation_missing,test_missing
event_name_1,5277819,85372,73176
event_type_1,5277819,85372,73176
event_name_2,5735169,85372,85372
event_type_2,5735169,85372,85372
sales_lag_28,85372,0,0
sales_lag_35,106715,0,0
sales_lag_42,128058,0,0
sales_lag_56,170744,0,0
rolling_mean_7_lag28,103666,0,0
rolling_mean_28_lag28,167695,0,0


In [49]:
# ---------------------------------------------------------
# Encode absence of calendar events explicitly
# ---------------------------------------------------------
# Missing event values do not represent unknown information.
# They mean that no corresponding event occurred on that date.
#
# We therefore convert missing event categories into an
# explicit "NoEvent" category rather than imputing them using
# the most frequent event or dropping those observations.

event_columns = [
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
]


for column in event_columns:

    # The columns currently use pandas categorical dtype.
    # A new category must be registered before missing values
    # can be filled with that label.
    if "NoEvent" not in df[column].cat.categories:

        df[column] = (
            df[column]
            .cat.add_categories(
                ["NoEvent"]
            )
        )

    df[column] = (
        df[column]
        .fillna("NoEvent")
    )

In [50]:
# ---------------------------------------------------------
# Recreate splits after event-category transformation
# ---------------------------------------------------------
# The existing split DataFrames are independent copies, so
# rebuilding them ensures they contain the updated event
# categories while preserving exactly the same date ranges.

train_df = (
    df.loc[
        df["date"].isin(training_dates)
    ]
    .copy()
)

validation_df = (
    df.loc[
        df["date"].isin(validation_dates)
    ]
    .copy()
)

test_df = (
    df.loc[
        df["date"].isin(test_dates)
    ]
    .copy()
)

In [51]:
# ---------------------------------------------------------
# Confirm that event missingness has been eliminated
# ---------------------------------------------------------

event_missing_validation = pd.DataFrame(
    {
        "train_missing": (
            train_df[event_columns]
            .isna()
            .sum()
        ),
        "validation_missing": (
            validation_df[event_columns]
            .isna()
            .sum()
        ),
        "test_missing": (
            test_df[event_columns]
            .isna()
            .sum()
        ),
    }
)

event_missing_validation

,train_missing,validation_missing,test_missing
event_name_1,0,0,0
event_type_1,0,0,0
event_name_2,0,0,0
event_type_2,0,0,0


In [52]:
# ---------------------------------------------------------
# Define horizon-safe historical demand features
# ---------------------------------------------------------
# These features require actual historical sales information.
# Missing values occur only near the beginning of each
# product's time series, where sufficient history does not
# yet exist.

historical_demand_features = [
    "sales_lag_28",
    "sales_lag_35",
    "sales_lag_42",
    "sales_lag_56",
    "rolling_mean_7_lag28",
    "rolling_mean_28_lag28",
    "rolling_std_7_lag28",
    "rolling_std_28_lag28",
]


# ---------------------------------------------------------
# Identify training rows with complete demand history
# ---------------------------------------------------------
# A row is retained only when every required historical
# demand feature is available.
#
# Validation and test are not filtered because our previous
# audit confirmed that these features are already complete
# in both future periods.

complete_demand_history = (
    train_df[
        historical_demand_features
    ]
    .notna()
    .all(axis=1)
)


rows_before_history_filter = len(
    train_df
)

train_df = (
    train_df.loc[
        complete_demand_history
    ]
    .copy()
)

rows_after_history_filter = len(
    train_df
)


print(
    "Training rows before filter:",
    f"{rows_before_history_filter:,}"
)

print(
    "Training rows after filter:",
    f"{rows_after_history_filter:,}"
)

print(
    "Training rows removed:",
    f"{rows_before_history_filter - rows_after_history_filter:,}"
)

print(
    "Percentage removed:",
    (
        f"{(rows_before_history_filter - rows_after_history_filter) / rows_before_history_filter * 100:.2f}%"
    )
)

print(
    "New training start date:",
    train_df["date"].min().date()
)

Training rows before filter: 5,747,365
Training rows after filter: 5,576,621
Training rows removed: 170,744
Percentage removed: 2.97%
New training start date: 2011-03-26


In [53]:
# ---------------------------------------------------------
# Re-audit price-feature missingness after history filtering
# ---------------------------------------------------------
# Removing the first 56 days per product eliminates structural
# missingness caused by insufficient demand history.
#
# Price missingness may still remain because missing M5 prices
# represent periods when a product was unavailable.
#
# We quantify that separately before deciding whether those
# observations should be retained, filtered, or handled by
# the modeling algorithm.

price_model_features = [
    "sell_price",
    "sell_price_lag_28",
    "price_diff_28",
    "log_price_ratio_28",
    "is_available",
]


price_missing_after_filter = pd.DataFrame(
    {
        "train_missing": (
            train_df[price_model_features]
            .isna()
            .sum()
        ),
        "train_missing_pct": (
            train_df[price_model_features]
            .isna()
            .mean()
            * 100
        ),
        "validation_missing": (
            validation_df[price_model_features]
            .isna()
            .sum()
        ),
        "test_missing": (
            test_df[price_model_features]
            .isna()
            .sum()
        ),
    }
)

price_missing_after_filter

,train_missing,train_missing_pct,validation_missing,test_missing
sell_price,1036749,18.590989,0,0
sell_price_lag_28,1081388,19.391456,0,0
price_diff_28,1081388,19.391456,0,0
log_price_ratio_28,1081395,19.391581,0,0
is_available,0,0.000000,0,0


In [54]:
# ---------------------------------------------------------
# Examine sales behavior during unavailable training periods
# ---------------------------------------------------------
# Earlier data preparation showed that unavailable observations
# had zero sales. We verify that this remains true within the
# filtered training dataset used for modeling.
#
# If unavailable rows always have zero sales, availability
# itself provides strong structural information about demand.

unavailable_train = (
    train_df["is_available"].eq(0)
)


print(
    "Unavailable training rows:",
    f"{unavailable_train.sum():,}"
)

print(
    "Unavailable rows with positive sales:",
    f"{(
        train_df.loc[unavailable_train, 'sales'] > 0
    ).sum():,}"
)

print(
    "Sales total during unavailable periods:",
    f"{train_df.loc[unavailable_train, 'sales'].sum():,}"
)

Unavailable training rows: 1,036,749
Unavailable rows with positive sales: 0
Sales total during unavailable periods: 0


### Availability-Aware Forecasting Strategy

Product availability is treated as a structural business constraint rather
than as ordinary demand behavior.

Within the filtered training data, every observation where
`is_available == 0` has exactly zero recorded sales. Therefore, unavailable
product-days do not represent low customer demand; they represent periods
during which the product could not generate sales.

The forecasting pipeline will consequently use an availability-aware rule:

1. If a product is unavailable for the target date, predicted sales are set
   deterministically to zero.
2. If a product is available, the machine-learning model predicts demand
   using historical demand, calendar, event, SNAP, product, and price
   information.

This prevents structurally unavailable observations from dominating the
zero-demand class and allows the model to focus on demand variation among
products that can actually be sold.

In [56]:
# ---------------------------------------------------------
# Restrict ML training to available product-days
# ---------------------------------------------------------
# Unavailable observations have a deterministic sales outcome
# of zero, so they do not need to be learned by the regression
# model.
#
# The production forecasting pipeline will handle those rows
# separately using the availability rule.

train_available = (
    train_df.loc[
        train_df["is_available"].eq(1)
    ]
    .copy()
)


# ---------------------------------------------------------
# Validate the available-product training population
# ---------------------------------------------------------

print(
    "Filtered training rows:",
    f"{len(train_df):,}"
)

print(
    "Available rows used for ML:",
    f"{len(train_available):,}"
)

print(
    "Unavailable rows handled by rule:",
    f"{len(train_df) - len(train_available):,}"
)

print(
    "Share used for ML:",
    f"{len(train_available) / len(train_df) * 100:.2f}%"
)

print(
    "Available training start:",
    train_available["date"].min().date()
)

print(
    "Available training end:",
    train_available["date"].max().date()
)

Filtered training rows: 5,576,621
Available rows used for ML: 4,539,872
Unavailable rows handled by rule: 1,036,749
Share used for ML: 81.41%
Available training start: 2011-03-26
Available training end: 2016-03-27


In [57]:
# ---------------------------------------------------------
# Final missing-value audit for the ML training population
# ---------------------------------------------------------
# At this point:
#
# - early observations without sufficient demand history have
#   already been removed;
# - unavailable product-days have been separated from the ML
#   training population;
# - event missingness has been encoded as "NoEvent".
#
# Ideally, the remaining available training observations
# should now have a complete model feature vector.

ml_missing_audit = pd.DataFrame(
    {
        "missing_rows": (
            train_available[model_features]
            .isna()
            .sum()
        ),
        "missing_pct": (
            train_available[model_features]
            .isna()
            .mean()
            * 100
        ),
    }
)


# Display only features that still contain missing values.
ml_missing_audit = (
    ml_missing_audit.loc[
        ml_missing_audit["missing_rows"] > 0
    ]
)

ml_missing_audit

,missing_rows,missing_pct
sell_price_lag_28,44639,0.983266
price_diff_28,44639,0.983266
log_price_ratio_28,44646,0.983420


In [58]:
# ---------------------------------------------------------
# Create historical-price availability indicator
# ---------------------------------------------------------
# Some products are available today but were unavailable
# 28 days earlier.
#
# Rather than dropping these legitimate demand observations,
# we explicitly tell the model whether a valid historical
# price comparison exists.

train_available["has_price_history_28"] = (
    train_available["sell_price_lag_28"]
    .notna()
    .astype("uint8")
)

In [59]:
# ---------------------------------------------------------
# Apply the same feature definition to future datasets
# ---------------------------------------------------------
# Feature engineering must be identical across training,
# validation, and test data.

for dataset in [
    validation_df,
    test_df,
]:

    dataset["has_price_history_28"] = (
        dataset["sell_price_lag_28"]
        .notna()
        .astype("uint8")
    )

In [60]:
# ---------------------------------------------------------
# Handle unavailable historical price comparisons
# ---------------------------------------------------------
# Derived price-change features are set to their neutral value
# when historical price information is unavailable.
#
# price_diff_28 = 0:
#     no measurable absolute historical change
#
# log_price_ratio_28 = 0:
#     no measurable proportional historical change
#
# has_price_history_28 allows the model to distinguish these
# imputed neutral values from genuine unchanged prices.

for dataset in [
    train_available,
    validation_df,
    test_df,
]:

    dataset["price_diff_28"] = (
        dataset["price_diff_28"]
        .fillna(0.0)
    )

    dataset["log_price_ratio_28"] = (
        dataset["log_price_ratio_28"]
        .fillna(0.0)
    )

In [61]:
# ---------------------------------------------------------
# Validate historical-price handling
# ---------------------------------------------------------

price_history_summary = pd.DataFrame(
    {
        "train": [
            train_available["has_price_history_28"].eq(0).sum(),
            train_available["price_diff_28"].isna().sum(),
            train_available["log_price_ratio_28"].isna().sum(),
        ],

        "validation": [
            validation_df["has_price_history_28"].eq(0).sum(),
            validation_df["price_diff_28"].isna().sum(),
            validation_df["log_price_ratio_28"].isna().sum(),
        ],

        "test": [
            test_df["has_price_history_28"].eq(0).sum(),
            test_df["price_diff_28"].isna().sum(),
            test_df["log_price_ratio_28"].isna().sum(),
        ],
    },
    index=[
        "missing_price_history_28",
        "missing_price_diff_28",
        "missing_log_price_ratio_28",
    ],
)

price_history_summary

,train,validation,test
missing_price_history_28,44639,0,0
missing_price_diff_28,0,0,0
missing_log_price_ratio_28,0,0,0


In [62]:
# ---------------------------------------------------------
# Finalize the numeric model feature set
# ---------------------------------------------------------
# sell_price_lag_28 is excluded from the final model because
# some legitimate available observations have no historical
# price 28 days earlier.
#
# Historical price information is instead represented by:
#
#   price_diff_28
#   log_price_ratio_28
#   has_price_history_28
#
# This avoids inventing a raw historical selling price.

numeric_features = [
    # Calendar features
    "wday",
    "month",
    "year",
    "wm_yr_wk",
    "snap_CA",
    "day_of_month",
    "week_of_year",
    "quarter",
    "is_weekend",
    "is_event",

    # Horizon-safe historical demand features
    "sales_lag_28",
    "sales_lag_35",
    "sales_lag_42",
    "sales_lag_56",
    "rolling_mean_7_lag28",
    "rolling_mean_28_lag28",
    "rolling_std_7_lag28",
    "rolling_std_28_lag28",

    # Price and availability features
    "sell_price",
    "price_diff_28",
    "log_price_ratio_28",
    "has_price_history_28",
    "is_available",
]


# Rebuild the complete model feature list.
model_features = (
    categorical_features
    + numeric_features
)


print(
    "Categorical features:",
    len(categorical_features)
)

print(
    "Numeric features:",
    len(numeric_features)
)

print(
    "Total model features:",
    len(model_features)
)

Categorical features: 8
Numeric features: 23
Total model features: 31


In [63]:
# ---------------------------------------------------------
# Final model-input completeness audit
# ---------------------------------------------------------
# Before constructing X/y matrices, every feature used by the
# ML model should be present and complete.
#
# Validation and test are checked using all rows because the
# availability business rule will later be applied when
# generating predictions.

final_feature_audit = pd.DataFrame(
    {
        "train_missing": (
            train_available[model_features]
            .isna()
            .sum()
        ),

        "validation_missing": (
            validation_df[model_features]
            .isna()
            .sum()
        ),

        "test_missing": (
            test_df[model_features]
            .isna()
            .sum()
        ),
    }
)


# Display only problematic features.
final_feature_audit = (
    final_feature_audit.loc[
        final_feature_audit.sum(axis=1) > 0
    ]
)


print(
    "Total model features:",
    len(model_features)
)

print(
    "Features with remaining missing values:",
    len(final_feature_audit)
)

final_feature_audit

Total model features: 31
Features with remaining missing values: 0


,train_missing,validation_missing,test_missing
